# Note 5.1 : Improving integration accuracy by boundary values

## Motivation  

Consider an integral 
$I(a,b) = \int_a^b dx f(x)$.  
We discretize the integration domain into N parts:  
$x_0=a, x_1=a+h, \dots, x_k=a+kh, \dots, x_N=b$ with $h=(b-a)/N$.  

The most naive formula is approximating the integral by sum of rectangles,  
$I_{\rm rec} \equiv h\sum_{k=1}^N f(x_k)$.  
Using trapezoids instead of rectangles generally improves the accuracy, and the trapezoid method turns out to simply add boundary values to $I_{\rm rec}$,  
$I_{\rm trapz} \equiv I_{\rm rec} - \frac{1}{2}h(f(b)-f(a))$.  
The error of $I_{\rm trapz}$ is $\mathcal O(h^2)$,  
$I(a,b) = I_{\rm trapz} + \mathcal O(h^2)$.  
Adding the Euler-Maclaurin term to $I_{\rm trapz}$,  
$I_{\rm em} \equiv I_{\rm trapz} + \frac{1}{12}h^2(f'(b)-f'(a))$,  
further reduces the error to $\mathcal O(h^4)$, that is,  
$I(a,b) = I_{\rm em} + \mathcal O(h^4)$. 

I felt the fact above somewhat miraculous;  integration is inherently a __bulk__ operation ― it measures the area under the curve $f(x)$ between $a$ and $b$.  But the fact above says that we can reduce the error of this bulk quantity by only adding __boundary__ values, $f(a), f(b), f'(a), f'(b)$ and so on. 

So, I want to check by concrete examples that the errors of rectangular method, the trapezoidal method, and the trapezoidal method with the Euler-Maclaurin term are indeed of order $\mathcal O(h), \mathcal O(h^2), \mathcal O(h^4)$, respectively.  

## Task  

Numerically calculate each of $I_{\rm rec}, I_{\rm trapz}, I_{\rm em}$ for various integrations, and check if  
 - the errors of $I_{\rm rec}, I_{\rm trapz}$ and $I_{\rm em}$ are $\mathcal O(h), \mathcal O(h^2)$ and $\mathcal O(h^4)$, respectively.

I will do this by the following steps.  We assume that we know the true value of an integral $I(a,b)$.  
 - Assume that a numerical integration value $I^{(h)}$ with step size $h$ has error of order $\gamma$, that is, $I^{(h)}=I(a,b)+ch^\gamma \cdots$, where $c$ is some constant and "$\cdots$" represents terms with order higher than $\gamma$ and are assumed negligible.
 - Compute $I^{(h)}$ and $I^{(h/2)}$, where the latter is obtained by halving the step size from $h$ to $\frac{h}{2}$.
 - Since $I^{(h)}=I(a,b)+ch^\gamma$ and $I^{(h/2)}=I(a,b)+c\left(\frac{h}{2}\right)^\gamma$, then
$$
\frac{I^{(h)}-I(a,b)}{I^{(h/2)}-I(a,b)}= \frac{ch^\gamma}{c\left(\frac{h}{2}\right)^\gamma}=2^\gamma.
$$ So we can obtain the value $\gamma$ by taking $\log_2$ of this.  

In [1]:
import math

def calculate_integral(f, df, a, b, N, method):
    """
    Returns integral value of a function f from a to b.  
    N represents the number of division of the integral domain [a, b]
    "method" is a string selecting a integration method from "rectangle", "trapezoid", "euler-maclaurin" 
    df is the derivative of f, which is needed if we choose "euler-maclaurin" as the method.
    """
    assert a<b
    assert method in ["rectangle", "trapezoid", "euler-maclaurin"]
    
    h = (b-a)/N
    
    result = 0.
    for k in range(1,N+1):
        x_k = a + h*k
        result += h*f(x_k)
    if method == "rectangle":
        return result
    result += 0.5*h*(f(a) - f(b))  # trapezoid
    if method == "trapezoid":
        return result
    result += 1/12*h*h*(df(a) - df(b))  # euler-maclaurin
    return result

def print_result(f, df, a, b, true_value, N):
    """
    this is a wrapper of the function 'calculate_integral'.
    true_value is the analytical result of the integral you should perform by hand beforehand.
    For each method of "rectangle", "trapezoid", "euler-maclaurin", 
    this function prints 
       the numerically calculated integral value
       the error (i.e. the absolute difference from the true_value)
       the order of the error, i.e., log_h(the error)
    """
    print(f"true_value:{true_value}")
    h = (b-a)/N
    for method in ["rectangle", "trapezoid", "euler-maclaurin"]:
        integral = calculate_integral(f, df, a, b, N, method=method)
        error = abs((integral - true_value)/true_value)
        #error_order = math.log(error, h)

        integral_2 = calculate_integral(f, df, a, b, 2*N, method=method)
        error_2 = abs((integral_2 - true_value)/true_value)
        #error_change_rate = math.log(error/error_2, 2)
        error_order = math.log(error/error_2, 2)
        
        space = ""
        if method in ["rectangle", "trapezoid"]:
            space += " "*(len("euler-maclaurin") - len("rectangle"))  # just to align printing
        print(f"method:{method},{space} integral:{integral:.8f}, relative error:{error:.16f}, error_order:{error_order:.4f}")

$\int_0^2 dx (x^4 - 2x + 1) = \frac{22}{5}$  

In [2]:
print_result(lambda x:x**4-2*x+1, lambda x:4*x**3-2, a=0, b=2, true_value=22/5, N=100)

true_value:4.4
method:rectangle,       integral:4.52106666, relative error:0.0275151490909089, error_order:1.0064
method:trapezoid,       integral:4.40106666, relative error:0.0002424218181816, error_order:2.0000
method:euler-maclaurin, integral:4.39999999, relative error:0.0000000024242426, error_order:4.0000


In [3]:
print_result(lambda x:x**4-2*x+1, lambda x:4*x**3-2, a=0, b=2, true_value=22/5, N=1000)  # changed N to 1000

true_value:4.4
method:rectangle,       integral:4.41201067, relative error:0.0027296969694542, error_order:1.0006
method:trapezoid,       integral:4.40001067, relative error:0.0000024242421815, error_order:2.0000
method:euler-maclaurin, integral:4.40000000, relative error:0.0000000000002426, error_order:4.0613


$\int_0^{\pi/2} dx \sin x = 1$  

In [4]:
print_result(lambda x:math.sin(x), lambda x:math.cos(x), a=0, b=math.pi/2, true_value=1., N=100)

true_value:1.0
method:rectangle,       integral:1.00783342, relative error:0.0078334198735823, error_order:0.9981
method:trapezoid,       integral:0.99997944, relative error:0.0000205617603921, error_order:2.0000
method:euler-maclaurin, integral:1.00000000, relative error:0.0000000000845565, error_order:3.9999


In [5]:
print_result(lambda x:math.sin(x), lambda x:math.cos(x), a=0, b=math.pi/2, true_value=1., N=1000)  # changed N to 1000

true_value:1.0
method:rectangle,       integral:1.00078519, relative error:0.0007851925466311, error_order:0.9998
method:trapezoid,       integral:0.99999979, relative error:0.0000002056167663, error_order:2.0000
method:euler-maclaurin, integral:1.00000000, relative error:0.0000000000000079, error_order:2.8278


$\int_0^{\pi} dx \sin x = 2$ 

In [6]:
print_result(lambda x:math.sin(x), lambda x:math.cos(x), a=0, b=math.pi, true_value=2., N=100)

true_value:2.0
method:rectangle,       integral:1.99983550, relative error:0.0000822480562782, error_order:2.0000
method:trapezoid,       integral:1.99983550, relative error:0.0000822480562782, error_order:2.0000
method:euler-maclaurin, integral:2.00000000, relative error:0.0000000013529358, error_order:4.0000


In [7]:
print_result(lambda x:math.sin(x), lambda x:math.cos(x), a=0, b=math.pi, true_value=2., N=1000) # changed N to 1000

true_value:2.0
method:rectangle,       integral:1.99999836, relative error:0.0000008224671681, error_order:2.0000
method:trapezoid,       integral:1.99999836, relative error:0.0000008224671681, error_order:2.0000
method:euler-maclaurin, integral:2.00000000, relative error:0.0000000000001347, error_order:4.0349


$\int_1^3 dx \frac{1}{x} = \log 3$

In [8]:
print_result(lambda x:1/x, lambda x:-1/(x*x), a=1, b=3, true_value=math.log(3), N=100)

true_value:1.0986122886681098
method:rectangle,       integral:1.09197525, relative error:0.0060412926581249, error_order:0.9968
method:trapezoid,       integral:1.09864192, relative error:0.0000269688527206, error_order:2.0000
method:euler-maclaurin, integral:1.09861229, relative error:0.0000000011984387, error_order:3.9998


In [9]:
print_result(lambda x:1/x, lambda x:-1/(x*x), a=1, b=3, true_value=math.log(3), N=1000) # changed N to 1000

true_value:1.0986122886681098
method:rectangle,       integral:1.09794592, relative error:0.0006065564506933, error_order:0.9997
method:trapezoid,       integral:1.09861258, relative error:0.0000002697003912, error_order:2.0000
method:euler-maclaurin, integral:1.09861229, relative error:0.0000000000001205, error_order:3.9338


$\int_1^{100} dx \frac{1}{x} = \log 100$

In [14]:
print_result(lambda x:1/x, lambda x:-1/(x*x), a=1, b=100, true_value=math.log(100), N=100)  

true_value:4.605170185988092
method:rectangle,       integral:4.19088407, relative error:0.0899610880748861, error_order:0.8803
method:trapezoid,       integral:4.68093407, relative error:0.0164519173534582, error_order:1.9241
method:euler-maclaurin, integral:4.59926723, relative error:0.0012818100011753, error_order:3.7047


In [15]:
print_result(lambda x:1/x, lambda x:-1/(x*x), a=1, b=100, true_value=math.log(100), N=1000)  # changed N to 1000

true_value:4.605170185988092
method:rectangle,       integral:4.55698106, relative error:0.0104641362918649, error_order:0.9878
method:trapezoid,       integral:4.60598606, relative error:0.0001771642509696, error_order:1.9989
method:euler-maclaurin, integral:4.60516939, relative error:0.0000001730225768, error_order:3.9950


$\int_0^3 dx e^x = e^3-1$

In [12]:
print_result(lambda x:math.exp(x), lambda x:math.exp(x), a=0, b=3, true_value=math.exp(3)-1, N=100)

true_value:19.085536923187668
method:rectangle,       integral:19.37325137, relative error:0.0150749988750240, error_order:1.0036
method:trapezoid,       integral:19.08696832, relative error:0.0000749988750241, error_order:2.0000
method:euler-maclaurin, integral:19.08553690, relative error:0.0000000011249759, error_order:4.0000


In [13]:
print_result(lambda x:math.exp(x), lambda x:math.exp(x), a=0, b=3, true_value=math.exp(3)-1, N=1000)  # changed N to 1000

true_value:19.085536923187668
method:rectangle,       integral:19.11417954, relative error:0.0015007499998865, error_order:1.0004
method:trapezoid,       integral:19.08555124, relative error:0.0000007499998865, error_order:2.0000
method:euler-maclaurin, integral:19.08553692, relative error:0.0000000000001135, error_order:4.1234


Thus, we have observed that the error order $\gamma$ in the error term $I^{(h)}=I(a,b)+c h^\gamma$, agrees with theoretical expectation for each method.  

minor remark:  
In the case of $\int_0^{\pi} dx \sin x = 2$, the error order of the rectangle method is already $\gamma=2$, the same value of the trapezoidal rule.  This is because in this case $f(a)=f(b)=0$, and the rectangle method and the trapezoidal method give the same integral value.  